# Gold validation with Great Expectations (streaming)

Purpose: validate the Gold serving stream incrementally, enforce business-facing data contracts, and publish append-only observability outputs plus validated/quarantine sinks.


In [0]:
%pip install great-expectations==0.18.21


  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached makefun-1.16.0-py2.py3-none-any.whl.metadata (2.9 kB)


  Using cached ruamel.yaml-0.17.40-py3-none-any.whl.metadata (19 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)


  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached entrypoints-0.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
Using cached altair-4.2.2-py3-none-any.whl (813 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached makefun-1.16.0-py2.py3-none-any.whl (22 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached ruamel.yaml-0.17.40-py3-none-any.whl (113 kB)
Using cached tzlocal-5.3.1-py3-none-any.whl (18 kB)
Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (788 kB)
Using cached entrypoints-0.4-py3-none-any.whl (5.3 kB)
Using cach

In [0]:
import json
import os
import traceback
from datetime import datetime, timezone
from functools import reduce

import great_expectations as gx
from great_expectations.data_context import FileDataContext
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

STORAGE_ACCOUNT = "streanmingdatasta"
LAKEHOUSE_CONTAINER = "lakehouse"
ROOT_BASE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog"
GOLD_BASE_PATH = f"{ROOT_BASE_PATH}/gold"

GOLD_STREAM_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_vehicle_positions_serving_stream"
GOLD_VALIDATED_OUTPUT_PATH = f"{GOLD_BASE_PATH}/gold_validated_stream/hsl_vehicle_positions"
GOLD_QUARANTINE_OUTPUT_PATH = f"{GOLD_BASE_PATH}/gold_ge_quarantine_stream/hsl_vehicle_positions"
GOLD_GE_RESULTS_PATH = f"{GOLD_BASE_PATH}/gold_ge/ge_results/hsl_vehicle_positions"
GOLD_GE_DETAILS_PATH = f"{GOLD_BASE_PATH}/gold_ge/ge_details/hsl_vehicle_positions"
GOLD_FAILED_SAMPLES_PATH = f"{GOLD_BASE_PATH}/gold_ge/failed_row_samples/hsl_vehicle_positions"
GOLD_RULE_METRICS_PATH = f"{GOLD_BASE_PATH}/gold_ge/rule_metrics/hsl_vehicle_positions"
GOLD_ALERTS_PATH = f"{GOLD_BASE_PATH}/gold_ge/alerts/hsl_vehicle_positions"
GOLD_GATE_RESULTS_PATH = f"{GOLD_BASE_PATH}/gold_ge/gate_results/hsl_vehicle_positions"
CHECKPOINT_PATH = f"{GOLD_BASE_PATH}/checkpoints/gold_ge_validate_stream_hsl_vehicle_positions"

GOLD_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_vehicle_positions_validated_stream"
GOLD_VALIDATION_QUAR_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_vehicle_positions_ge_quarantine_stream"
GOLD_GE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_ge_results_hsl_vehicle_positions"
GOLD_GE_DETAILS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_ge_details_hsl_vehicle_positions"
GOLD_GE_SAMPLES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_ge_failed_samples_hsl_vehicle_positions"
GOLD_RULE_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_ge_rule_metrics_hsl_vehicle_positions"
GOLD_ALERTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_ge_alerts_hsl_vehicle_positions"
GOLD_GATE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_gate_results_hsl_vehicle_positions"

TRIGGER_INTERVAL = "10 seconds"
MAX_FAILED_SAMPLE_ROWS = 500
MAX_CRITICAL_ROWS = 0
MAX_HIGH_ROWS = 0
MAX_QUARANTINE_RATE = 0.01
MAX_MEDIUM_WARNING_RATE = 0.05

CONTEXT_ROOT_DIR = "/dbfs/great_expectations/gold_validate_ge_stream_ctx"
DATASOURCE_NAME = "gold_runtime_spark_stream"
DATA_ASSET_NAME = "gold_vehicle_positions_serving_stream_asset"
EXPECTATION_SUITE_NAME = "gold_serving_validation_stream_suite"
QUERY_NAME = "gold_vehicle_positions_ge_validate_stream"


def filter_to_target_ingest_date(df):
    if "ingest_date" not in df.columns:
        raise ValueError("Expected ingest_date column before validation date filtering.")
    return df.where(F.to_date(F.col("ingest_date")) == F.current_date())


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name == QUERY_NAME:
        q.stop()

base_schema = spark.table(GOLD_STREAM_TABLE).schema
sink_schema = StructType(list(base_schema.fields) + [
    StructField("gold_failed_rule_ids", ArrayType(StringType()), True),
    StructField("gold_failed_severities", ArrayType(StringType()), True),
    StructField("gold_failed_rule_descriptions", ArrayType(StringType()), True),
    StructField("gold_failed_rule_count", LongType(), True),
    StructField("gold_validation_run_id", StringType(), True),
    StructField("gold_validation_batch_id", LongType(), True),
    StructField("gold_validation_scope", StringType(), True),
    StructField("gold_validation_ts", TimestampType(), True),
    StructField("gold_validation_date", StringType(), True),
    StructField("gold_validation_status", StringType(), True),
])

def precreate_sink(path: str, table_name: str):
    if not DeltaTable.isDeltaTable(spark, path):
        (
            spark.createDataFrame([], sink_schema)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ingest_date")
            .save(path)
        )
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')

precreate_sink(GOLD_VALIDATED_OUTPUT_PATH, GOLD_VALIDATED_TABLE)
precreate_sink(GOLD_QUARANTINE_OUTPUT_PATH, GOLD_VALIDATION_QUAR_TABLE)


In [0]:
gate_schema = StructType([
    StructField("run_id", StringType()),
    StructField("validated_table", StringType()),
    StructField("run_ts_utc", TimestampType()),
    StructField("validation_date", StringType()),
    StructField("validation_scope", StringType()),
    StructField("input_row_count", LongType()),
    StructField("validated_row_count", LongType()),
    StructField("quarantined_row_count", LongType()),
    StructField("quarantine_rate", DoubleType()),
    StructField("critical_failed_rows", LongType()),
    StructField("high_failed_rows", LongType()),
    StructField("medium_failed_rows", LongType()),
    StructField("low_failed_rows", LongType()),
    StructField("warning_rate", DoubleType()),
    StructField("gate_status", StringType()),
    StructField("gate_reason", StringType()),
    StructField("serving_allowed", BooleanType()),
    StructField("validated_output_path", StringType()),
    StructField("quarantine_output_path", StringType()),
])


def register_table(path: str, table_name: str):
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')


def write_append(df, path: str, table_name: str, merge_schema: bool = True, partition_cols: list[str] | None = None):
    if not DeltaTable.isDeltaTable(spark, path):
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    else:
        writer = df.write.format("delta").mode("append")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    register_table(path, table_name)


def write_gate_results(rows):
    write_append(
        spark.createDataFrame(rows, schema=gate_schema),
        GOLD_GATE_RESULTS_PATH,
        GOLD_GATE_RESULTS_TABLE,
        merge_schema=False,
    )


def union_by_name_allow_missing(dfs):
    if not dfs:
        return None
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), dfs)


def gx_to_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "to_json_dict"):
        return obj.to_json_dict()
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    return {}


In [0]:
def get_gx_context():
    os.makedirs(CONTEXT_ROOT_DIR, exist_ok=True)
    print(json.dumps({
        "layer": "gold_validate_ge",
        "message": "Initializing Great Expectations context",
        "context_root_dir": CONTEXT_ROOT_DIR,
    }, default=str))
    context = FileDataContext.create(project_root_dir=CONTEXT_ROOT_DIR)
    return context


def apply_gold_expectations(validator):
    for column_name in [
        "event_ts", "vehicle_id", "business_key", "route_id", "direction_id", "line_id",
        "transport_mode", "latitude", "longitude", "speed", "delay_sec", "occupancy",
        "service_date", "canonical_route_id", "canonical_direction_id",
        "route_direction_key", "route_vehicle_key", "delay_status", "occupancy_status",
        "location_quality", "is_delayed", "gold_publish_ts", "gold_service_ts",
        "gold_event_date", "silver_validation_run_id",
    ]:
        validator.expect_column_to_exist(column_name)

    validator.expect_column_values_to_not_be_null("event_ts")
    validator.expect_column_values_to_not_be_null("vehicle_id")
    validator.expect_column_values_to_not_be_null("business_key")
    validator.expect_column_values_to_not_be_null("canonical_route_id")
    validator.expect_column_values_to_not_be_null("canonical_direction_id")
    validator.expect_column_values_to_not_be_null("route_direction_key")
    validator.expect_column_values_to_not_be_null("route_vehicle_key")
    validator.expect_column_values_to_not_be_null("service_date")
    validator.expect_column_values_to_not_be_null("gold_publish_ts")
    validator.expect_column_values_to_not_be_null("gold_service_ts")
    validator.expect_column_values_to_not_be_null("gold_event_date")
    validator.expect_column_values_to_not_be_null("delay_status")
    validator.expect_column_values_to_not_be_null("location_quality")
    validator.expect_column_values_to_not_be_null("occupancy_status")
    validator.expect_column_values_to_not_be_null("transport_mode")

    validator.expect_column_values_to_be_in_set("transport_mode", ["bus", "tram", "train", "metro", "ferry", "ubus", "robot"])
    validator.expect_column_values_to_be_in_set("canonical_direction_id", ["1", "2"])
    validator.expect_column_values_to_be_in_set("delay_status", ["early", "on_time", "delayed", "severely_delayed", "unknown"], mostly=1.0)
    validator.expect_column_values_to_be_in_set("occupancy_status", ["empty_or_low", "moderate", "busy", "crowded", "unknown"], mostly=1.0)
    validator.expect_column_values_to_be_in_set("location_quality", ["ok", "missing", "out_of_bounds"], mostly=1.0)
    validator.expect_column_values_to_be_in_set("is_delayed", [True, False], mostly=1.0)

    validator.expect_column_values_to_match_regex("route_direction_key", r"^.+\|.+$")
    validator.expect_column_values_to_match_regex("route_vehicle_key", r"^.+\|.+$")

    validator.expect_column_values_to_be_between("latitude", min_value=59.0, max_value=61.5)
    validator.expect_column_values_to_be_between("longitude", min_value=23.0, max_value=26.5)
    validator.expect_column_values_to_be_between("speed", min_value=0, max_value=50)
    validator.expect_column_values_to_be_between("occupancy", min_value=0, max_value=100)
    validator.expect_column_values_to_be_unique("business_key")

    validator.expect_column_pair_values_to_be_equal("route_id", "canonical_route_id")
    validator.expect_column_pair_values_to_be_equal("direction_id", "canonical_direction_id")


def run_gold_ge_validation(batch_df, batch_run_id: str, batch_run_ts):
    context = get_gx_context()
    context.add_or_update_expectation_suite(expectation_suite_name=EXPECTATION_SUITE_NAME)
    datasource = context.sources.add_or_update_spark(name=DATASOURCE_NAME)
    asset = datasource.add_dataframe_asset(name=f"{DATA_ASSET_NAME}_{batch_run_id}")
    batch_request = asset.build_batch_request(dataframe=batch_df)
    validator = context.get_validator(batch_request=batch_request, expectation_suite_name=EXPECTATION_SUITE_NAME)
    apply_gold_expectations(validator)
    validator.save_expectation_suite(discard_failed_expectations=False)
    validation_result = validator.validate(
        run_id={"run_name": batch_run_id, "run_time": batch_run_ts.isoformat()},
        data_context=context,
    )
    return gx_to_dict(validation_result)


In [0]:
rule_specs = [
    {"rule_id": "critical_event_ts_null", "severity": "critical", "description": "Gold event timestamp must be present.", "fail_condition": F.col("event_ts").isNull(), "quarantine_row": True},
    {"rule_id": "critical_vehicle_id_null", "severity": "critical", "description": "Gold vehicle identifier must be present.", "fail_condition": F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == ""), "quarantine_row": True},
    {"rule_id": "critical_business_key_null", "severity": "critical", "description": "Business key must be present for serving contracts.", "fail_condition": F.col("business_key").isNull() | (F.trim(F.col("business_key")) == ""), "quarantine_row": True},
    {"rule_id": "high_route_id_null", "severity": "high", "description": "Canonical route identifier must be present.", "fail_condition": F.col("canonical_route_id").isNull() | (F.trim(F.col("canonical_route_id")) == ""), "quarantine_row": True},
    {"rule_id": "high_direction_invalid", "severity": "high", "description": "Canonical direction must be 1 or 2.", "fail_condition": F.col("canonical_direction_id").isNull() | ~F.col("canonical_direction_id").isin("1", "2"), "quarantine_row": True},
    {"rule_id": "high_service_date_null", "severity": "high", "description": "Service date must be present for business reporting.", "fail_condition": F.col("service_date").isNull(), "quarantine_row": True},
    {"rule_id": "high_coordinates_out_of_bounds", "severity": "high", "description": "Coordinates must stay within HSL operating bounds when present.", "fail_condition": (F.col("latitude").isNotNull() & ((F.col("latitude") < F.lit(59.0)) | (F.col("latitude") > F.lit(61.5)))) | (F.col("longitude").isNotNull() & ((F.col("longitude") < F.lit(23.0)) | (F.col("longitude") > F.lit(26.5)))), "quarantine_row": True},
    {"rule_id": "medium_transport_mode_unexpected", "severity": "medium", "description": "Transport mode should stay within the agreed domain.", "fail_condition": F.col("transport_mode").isNotNull() & ~F.col("transport_mode").isin("bus", "tram", "train", "metro", "ferry", "ubus", "robot"), "quarantine_row": True},
    {"rule_id": "medium_delay_status_invalid", "severity": "medium", "description": "Delay status should match the published business domain.", "fail_condition": F.col("delay_status").isNull() | ~F.col("delay_status").isin("early", "on_time", "delayed", "severely_delayed", "unknown"), "quarantine_row": True},
    {"rule_id": "medium_occupancy_status_invalid", "severity": "medium", "description": "Occupancy status should match the published business domain.", "fail_condition": F.col("occupancy_status").isNull() | ~F.col("occupancy_status").isin("empty_or_low", "moderate", "busy", "crowded", "unknown"), "quarantine_row": True},
    {"rule_id": "medium_negative_speed", "severity": "medium", "description": "Vehicle speed should not be negative.", "fail_condition": F.col("speed").isNotNull() & (F.col("speed") < F.lit(0.0)), "quarantine_row": True},
    {"rule_id": "low_location_quality_mismatch", "severity": "low", "description": "Location quality should align with coordinate availability.", "fail_condition": (F.col("has_coordinates") & (F.col("location_quality") == F.lit("missing"))) | (~F.col("has_coordinates") & (F.col("location_quality") == F.lit("ok"))), "quarantine_row": False},
    {"rule_id": "low_delay_flag_mismatch", "severity": "low", "description": "is_delayed should align with delay seconds over the 120-second threshold.", "fail_condition": F.col("delay_sec").isNotNull() & (F.col("is_delayed") != (F.col("delay_sec") > F.lit(120))), "quarantine_row": True},
]


In [0]:
def write_gold_validation_batch(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    batch_df = filter_to_target_ingest_date(batch_df)
    if batch_df.isEmpty():
        return

    batch_run_ts = datetime.now(timezone.utc)
    batch_run_id = f"gold_ge_stream_{batch_id}_{batch_run_ts.strftime('%Y%m%d_%H%M%S')}"
    validation_date = batch_run_ts.date().isoformat()
    validation_scope = f"microbatch_id={batch_id}"

    working_df = batch_df.cache()

    try:
        row_count = working_df.count()
        print(json.dumps({
            "layer": "gold_validate_ge",
            "message": "Starting foreachBatch validation",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "row_count": int(row_count),
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
        }, default=str))

        validation_result = run_gold_ge_validation(working_df, batch_run_id, batch_run_ts)
        print(json.dumps({
            "layer": "gold_validate_ge",
            "batch_run_id": batch_run_id,
        }, default=str))
        stats = validation_result.get("statistics", {}) or {}

        summary_row = [{
            "run_id": batch_run_id,
            "validated_table": GOLD_STREAM_TABLE,
            "validation_scope": validation_scope,
            "validation_date": validation_date,
            "run_ts_utc": batch_run_ts,
            "row_count": int(row_count),
            "success": bool(validation_result.get("success", False)),
            "evaluated_expectations": int(stats.get("evaluated_expectations", 0) or 0),
            "successful_expectations": int(stats.get("successful_expectations", 0) or 0),
            "unsuccessful_expectations": int(stats.get("unsuccessful_expectations", 0) or 0),
            "success_percent": float(stats.get("success_percent", 0.0) or 0.0),
        }]
        write_append(spark.createDataFrame(summary_row), GOLD_GE_RESULTS_PATH, GOLD_GE_RESULTS_TABLE)

        detail_rows = []
        for result in validation_result.get("results", []):
            cfg = result.get("expectation_config", {}) or {}
            detail_rows.append({
                "run_id": batch_run_id,
                "validated_table": GOLD_STREAM_TABLE,
                "validation_date": validation_date,
                "run_ts_utc": batch_run_ts,
                "expectation_type": cfg.get("expectation_type"),
                "column_name": (cfg.get("kwargs", {}) or {}).get("column"),
                "success": bool(result.get("success", False)),
                "result_json": json.dumps(result.get("result", {}), default=str),
                "kwargs_json": json.dumps((cfg.get("kwargs", {}) or {}), default=str),
            })
        if detail_rows:
            write_append(spark.createDataFrame(detail_rows), GOLD_GE_DETAILS_PATH, GOLD_GE_DETAILS_TABLE)

        failed_dfs = []
        for spec in rule_specs:
            failed_dfs.append(
                working_df
                .filter(spec["fail_condition"])
                .withColumn("rule_id", F.lit(spec["rule_id"]))
                .withColumn("severity", F.lit(spec["severity"]))
                .withColumn("rule_description", F.lit(spec["description"]))
                .withColumn("quarantine_row", F.lit(spec["quarantine_row"]))
                .withColumn("run_id", F.lit(batch_run_id))
                .withColumn("validation_date", F.lit(validation_date))
                .withColumn("validation_scope", F.lit(validation_scope))
                .withColumn("gold_validation_run_id", F.lit(batch_run_id))
                .withColumn("gold_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                .withColumn("gold_validation_scope", F.lit(validation_scope))
                .withColumn("gold_validation_ts", F.lit(batch_run_ts))
                .withColumn("gold_validation_date", F.lit(validation_date))
            )

        duplicate_keys_df = working_df.groupBy("business_key").count().filter(F.col("count") > 1).select("business_key")
        if duplicate_keys_df.count() > 0:
            failed_dfs.append(
                working_df.alias("g")
                .join(duplicate_keys_df.alias("d"), on=["business_key"], how="inner")
                .withColumn("rule_id", F.lit("critical_duplicate_business_key"))
                .withColumn("severity", F.lit("critical"))
                .withColumn("rule_description", F.lit("Business key must be unique within the Gold validation microbatch."))
                .withColumn("quarantine_row", F.lit(True))
                .withColumn("run_id", F.lit(batch_run_id))
                .withColumn("validation_date", F.lit(validation_date))
                .withColumn("validation_scope", F.lit(validation_scope))
                .withColumn("gold_validation_run_id", F.lit(batch_run_id))
                .withColumn("gold_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                .withColumn("gold_validation_scope", F.lit(validation_scope))
                .withColumn("gold_validation_ts", F.lit(batch_run_ts))
                .withColumn("gold_validation_date", F.lit(validation_date))
            )

        all_failed_rows_df = union_by_name_allow_missing(failed_dfs)
        if all_failed_rows_df is None:
            all_failed_rows_df = spark.createDataFrame(
                [],
                schema=(
                    working_df
                    .withColumn("rule_id", F.lit(None).cast("string"))
                    .withColumn("severity", F.lit(None).cast("string"))
                    .withColumn("rule_description", F.lit(None).cast("string"))
                    .withColumn("quarantine_row", F.lit(None).cast("boolean"))
                    .withColumn("run_id", F.lit(None).cast("string"))
                    .withColumn("validation_date", F.lit(None).cast("string"))
                    .withColumn("validation_scope", F.lit(None).cast("string"))
                    .withColumn("gold_validation_run_id", F.lit(None).cast("string"))
                    .withColumn("gold_validation_batch_id", F.lit(None).cast("bigint"))
                    .withColumn("gold_validation_scope", F.lit(None).cast("string"))
                    .withColumn("gold_validation_ts", F.lit(None).cast("timestamp"))
                    .withColumn("gold_validation_date", F.lit(None).cast("string"))
                    .schema
                ),
            )

        failed_rollup_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .groupBy("business_key")
            .agg(
                F.collect_set("rule_id").alias("gold_failed_rule_ids"),
                F.collect_set("severity").alias("gold_failed_severities"),
                F.collect_set("rule_description").alias("gold_failed_rule_descriptions"),
                F.max(F.when(F.col("quarantine_row") == True, F.lit(1)).otherwise(F.lit(0))).alias("should_quarantine"),
            )
        )

        empty_array = F.expr("array()").cast("array<string>")
        classified_df = (
            working_df
            .join(failed_rollup_df, on="business_key", how="left")
            .withColumn("gold_validation_run_id", F.lit(batch_run_id))
            .withColumn("gold_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
            .withColumn("gold_validation_scope", F.lit(validation_scope))
            .withColumn("gold_validation_ts", F.lit(batch_run_ts))
            .withColumn("gold_validation_date", F.lit(validation_date))
            .withColumn("gold_failed_rule_ids", F.when(F.col("gold_failed_rule_ids").isNull(), empty_array).otherwise(F.col("gold_failed_rule_ids")))
            .withColumn("gold_failed_severities", F.when(F.col("gold_failed_severities").isNull(), empty_array).otherwise(F.col("gold_failed_severities")))
            .withColumn("gold_failed_rule_descriptions", F.when(F.col("gold_failed_rule_descriptions").isNull(), empty_array).otherwise(F.col("gold_failed_rule_descriptions")))
            .withColumn("gold_failed_rule_count", F.size(F.col("gold_failed_rule_ids")).cast("bigint"))
            .withColumn(
                "gold_validation_status",
                F.when(F.coalesce(F.col("should_quarantine"), F.lit(0)) == F.lit(1), F.lit("quarantined"))
                 .otherwise(F.lit("validated"))
            )
            .drop("should_quarantine")
        )

        validated_df = classified_df.filter(F.col("gold_validation_status") == "validated").cache()
        quarantine_df = classified_df.filter(F.col("gold_validation_status") == "quarantined").cache()
        try:
            validated_row_count = validated_df.count()
            quarantined_row_count = quarantine_df.count()
            quarantine_rate = float(quarantined_row_count) / float(row_count) if row_count else 0.0
            write_append(validated_df, GOLD_VALIDATED_OUTPUT_PATH, GOLD_VALIDATED_TABLE, merge_schema=False, partition_cols=["ingest_date"])
            write_append(quarantine_df, GOLD_QUARANTINE_OUTPUT_PATH, GOLD_VALIDATION_QUAR_TABLE, merge_schema=False, partition_cols=["ingest_date"])
        finally:
            validated_df.unpersist()
            quarantine_df.unpersist()

        rule_metrics_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .groupBy("run_id", "validation_date", "validation_scope", "rule_id", "severity", "rule_description")
            .agg(F.count("*").alias("failed_rule_rows"), F.countDistinct("business_key").alias("affected_rows"))
        )
        if not rule_metrics_df.isEmpty():
            write_append(rule_metrics_df, GOLD_RULE_METRICS_PATH, GOLD_RULE_METRICS_TABLE)

        failed_samples_df = all_failed_rows_df.filter(F.col("rule_id").isNotNull()).limit(MAX_FAILED_SAMPLE_ROWS)
        if not failed_samples_df.isEmpty():
            write_append(failed_samples_df, GOLD_FAILED_SAMPLES_PATH, GOLD_GE_SAMPLES_TABLE)

        alert_rows = []
        for row in rule_metrics_df.collect():
            if row["severity"] in ("critical", "high"):
                alert_rows.append({
                    "run_id": row["run_id"],
                    "alert_ts": batch_run_ts,
                    "severity": row["severity"],
                    "rule_id": row["rule_id"],
                    "message": f"{row['rule_id']} affected {row['affected_rows']} Gold rows",
                    "failed_rows": int(row["affected_rows"]),
                })
        if alert_rows:
            write_append(spark.createDataFrame(alert_rows), GOLD_ALERTS_PATH, GOLD_ALERTS_TABLE)

        severity_rollup = {
            row["severity"]: row["affected_rows"]
            for row in (
                all_failed_rows_df
                .filter(F.col("rule_id").isNotNull())
                .select("severity", "business_key")
                .distinct()
                .groupBy("severity")
                .agg(F.count("*").alias("affected_rows"))
                .collect()
            )
        }

        critical_failed_rows = int(severity_rollup.get("critical", 0))
        high_failed_rows = int(severity_rollup.get("high", 0))
        medium_failed_rows = int(severity_rollup.get("medium", 0))
        low_failed_rows = int(severity_rollup.get("low", 0))
        warning_rows = medium_failed_rows + low_failed_rows
        warning_rate = float(warning_rows) / float(row_count) if row_count else 0.0
        print(json.dumps({
            "layer": "gold_validate_ge",
            "message": "Batch classification summary",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "row_count": int(row_count),
            "validated_row_count": int(validated_row_count),
            "quarantined_row_count": int(quarantined_row_count),
            "quarantine_rate": float(quarantine_rate),
            "critical_failed_rows": int(critical_failed_rows),
            "high_failed_rows": int(high_failed_rows),
            "medium_failed_rows": int(medium_failed_rows),
            "low_failed_rows": int(low_failed_rows),
        }, default=str))

        block_reasons = []
        warn_reasons = []
        if critical_failed_rows > MAX_CRITICAL_ROWS:
            block_reasons.append(f"critical_failed_rows={critical_failed_rows} exceeds threshold {MAX_CRITICAL_ROWS}")
        if high_failed_rows > MAX_HIGH_ROWS:
            block_reasons.append(f"high_failed_rows={high_failed_rows} exceeds threshold {MAX_HIGH_ROWS}")
        if quarantine_rate > MAX_QUARANTINE_RATE:
            block_reasons.append(f"quarantine_rate={quarantine_rate:.6f} exceeds threshold {MAX_QUARANTINE_RATE:.6f}")
        if medium_failed_rows > 0:
            warn_reasons.append(f"medium_failed_rows={medium_failed_rows}")
        if low_failed_rows > 0:
            warn_reasons.append(f"low_failed_rows={low_failed_rows}")
        if warning_rate > MAX_MEDIUM_WARNING_RATE:
            warn_reasons.append(f"warning_rate={warning_rate:.6f} exceeds threshold {MAX_MEDIUM_WARNING_RATE:.6f}")

        if block_reasons:
            gate_status = "BLOCK"
            serving_allowed = False
            gate_reason = "; ".join(block_reasons)
        elif warn_reasons:
            gate_status = "WARN"
            serving_allowed = True
            gate_reason = "; ".join(warn_reasons)
        else:
            gate_status = "PASS"
            serving_allowed = True
            gate_reason = "No failed rows detected above warning threshold."

        write_gate_results([{
            "run_id": batch_run_id,
            "validated_table": GOLD_STREAM_TABLE,
            "run_ts_utc": batch_run_ts,
            "validation_date": validation_date,
            "validation_scope": validation_scope,
            "input_row_count": int(row_count),
            "validated_row_count": int(validated_row_count),
            "quarantined_row_count": int(quarantined_row_count),
            "quarantine_rate": float(quarantine_rate),
            "critical_failed_rows": int(critical_failed_rows),
            "high_failed_rows": int(high_failed_rows),
            "medium_failed_rows": int(medium_failed_rows),
            "low_failed_rows": int(low_failed_rows),
            "warning_rate": float(warning_rate),
            "gate_status": gate_status,
            "gate_reason": gate_reason,
            "serving_allowed": bool(serving_allowed),
            "validated_output_path": GOLD_VALIDATED_OUTPUT_PATH,
            "quarantine_output_path": GOLD_QUARANTINE_OUTPUT_PATH,
        }])

        try:
            dbutils.jobs.taskValues.set(key="gold_gate_status", value=gate_status)
            dbutils.jobs.taskValues.set(key="gold_gate_reason", value=gate_reason[:1000])
            dbutils.jobs.taskValues.set(key="gold_validated_path", value=GOLD_VALIDATED_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(key="gold_quarantine_path", value=GOLD_QUARANTINE_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(key="gold_validation_run_id", value=batch_run_id)
        except Exception:
            pass
    except Exception as exc:
        print(json.dumps({
            "layer": "gold_validate_ge",
            "message": "foreachBatch failed",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "traceback": traceback.format_exc(),
        }, default=str))
        raise
    finally:
        working_df.unpersist()


In [0]:
gold_validation_query = (
    spark.readStream.table(GOLD_STREAM_TABLE)
    .writeStream
    .queryName(QUERY_NAME)
    .foreachBatch(write_gold_validation_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)


In [0]:
for q in spark.streams.active:
    if q.name == QUERY_NAME:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        break

gold_validation_query.awaitTermination()


NAME: gold_vehicle_positions_ge_validate_stream
ID: 3deccc51-eb4a-4c3a-93e4-9d2a08c6691b
IS ACTIVE: True
STATUS: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
{"layer": "gold_validate_ge", "message": "Starting foreachBatch validation", "batch_id": 3, "batch_run_id": "gold_ge_stream_3_20260422_051723", "row_count": 100354, "validation_scope": "microbatch_id=3", "context_root_dir": "/dbfs/great_expectations/gold_validate_ge_stream_ctx"}
{"layer": "gold_validate_ge", "message": "Initializing Great Expectations context", "context_root_dir": "/dbfs/great_expectations/gold_validate_ge_stream_ctx"}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-f8fb1280-57f2-49b4-b8a3-7eb61db2d930/lib/python3.12/site-packages/great_expectations/expectations/expectation.py:1519: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/153 [00:00<?, ?it/s]

{"layer": "gold_validate_ge", "batch_run_id": "gold_ge_stream_6_20260422_061911"}
{"layer": "gold_validate_ge", "message": "Batch classification summary", "batch_id": 6, "batch_run_id": "gold_ge_stream_6_20260422_061911", "row_count": 283194, "validated_row_count": 283194, "quarantined_row_count": 0, "quarantine_rate": 0.0, "critical_failed_rows": 0, "high_failed_rows": 283194, "medium_failed_rows": 0, "low_failed_rows": 0}


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can